# 02 - Create the provisional U.S. cohort

**Reuse your completed global cleaning.** The original `notebooks/02_clean_data.ipynb` already normalizes text, audits duplicates and writes the globally cleaned 17-column dataset. Running that same 80-cell cleaning process in another copy would create drift. Here we only (1) apply the documented geographic proxy, (2) validate that the subset stays clean, and (3) save it under `data/Processed-Data/us_only/`.

This **does not** rewrite global `01-05b`. Country is *not verified*: the scope is U.S. state/DC suffix candidates.

In [1]:
from pathlib import Path
import sys
HERE = Path.cwd()
PROJECT_ROOT = next((p for p in [HERE, *HERE.parents] if (p / "utils" / "preprocessing.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open Jupyter from the repository root or notebooks directory")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
GLOBAL_DIR = PROJECT_ROOT / "data" / "Processed-Data"
US_DIR = GLOBAL_DIR / "us_only"

import pandas as pd
from utils.us_scope import us_state_proxy_mask, us_location_audit
source = GLOBAL_DIR / "DSI_kickstarterscrape_dataset_cleaned.csv"
if not source.is_file():
    raise FileNotFoundError(f"Missing {source}; restore the global cleaned CSV or run global 02")
global_clean = pd.read_csv(source)
assert global_clean["project id"].is_unique
assert global_clean.shape[1] == 17, "Unexpected schema; review changes to the global cleaning notebook"
audit, excluded_suffixes = us_location_audit(global_clean)
display(audit)

,all_rows,candidate_us_rows,excluded_or_uncertain_rows,missing_locations,no_two_letter_suffix
0,45815,42175,3640,1318,3640


In [2]:
candidate_mask = us_state_proxy_mask(global_clean)
us_clean = global_clean.loc[candidate_mask].copy().reset_index(drop=True)
assert len(us_clean) > 0, "No state-code candidates found; inspect the location format"
assert us_clean["project id"].is_unique
assert us_state_proxy_mask(us_clean).all()
assert set(us_clean["project id"]).issubset(set(global_clean["project id"]))
print("Candidate U.S. campaigns, all statuses:", len(us_clean))
print("Candidate U.S. settled:", us_clean["status"].isin(["successful", "failed"]).sum())
display(us_clean["status"].value_counts(dropna=False))

Candidate U.S. campaigns, all statuses: 42175
Candidate U.S. settled: 38420


status
successful    21022
failed        17398
live           3695
canceled         56
suspended         4
Name: count, dtype: int64

In [3]:
US_DIR.mkdir(parents=True, exist_ok=True)
output = US_DIR / "DSI_kickstarterscrape_dataset_cleaned.csv"
us_clean.to_csv(output, index=False)
reloaded = pd.read_csv(output)
assert len(reloaded) == len(us_clean)
assert reloaded["project id"].is_unique
assert us_state_proxy_mask(reloaded).all()
print("Saved provisional state-suffix cohort:", output)
print("Global cleaned CSV was not modified:", source)

Saved provisional state-suffix cohort: /Users/annasehgal/State-Street-1C-kickstarter-crowdfunding-recommendation-engine/data/Processed-Data/us_only/DSI_kickstarterscrape_dataset_cleaned.csv
Global cleaned CSV was not modified: /Users/annasehgal/State-Street-1C-kickstarter-crowdfunding-recommendation-engine/data/Processed-Data/DSI_kickstarterscrape_dataset_cleaned.csv


**Handoff:** Run `03_preprocess_data.ipynb` to select successful/failed campaigns and restrict the *existing global official train/test split* to this U.S. candidate cohort. Do not use the U.S. final test outcomes for feature selection or tuning.